# RipeSense — fruit detection and spectral-data audit
Run on **Google Colab**, Python 3, CPU for the default path; choose **T4 GPU** for optional fine-tuning.
No local data or hardware is required. Our three demo apples (including the rotten apple) are not training data.

1. Install → mount Drive → run public-image detection and export RGB crops.
2. Optionally enable COCO fine-tuning and compare both detectors on the same validation subset.
3. Run the Mendeley inventory and sample-image inspection. **Spectral accuracy is withheld unless specimen identity, stage and wavelength mappings are verified.**
4. Read the demo handoff; artifacts are written to a timestamped Drive run directory.

Reference: Sujal's `src/RipeSense_Train.ipynb` uses dataset `y5h6v8w6ms`, version 2 and a `ripeness` filename filter. This notebook does not reuse its image-level splitting or infer labels from days.

**New: DeepHS five-fruit training** follows the YOLO cells and can also run independently. The Mendeley audit is optional; its training gate applies only to Mendeley.


In [ ]:
%pip -q install ultralytics==8.3.235 remotezip==0.12.3 scikit-learn pandas pillow matplotlib pyyaml joblib


In [ ]:
from pathlib import Path
import os, json, random, hashlib, shutil, re, io, zipfile, sys, subprocess
from datetime import datetime, timezone
from collections import Counter, defaultdict
import requests, numpy as np, pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = Path('/content/drive/MyDrive/RipeSense_Detection_Spectral') / RUN_ID
WORK = Path('/content/ripesense_detection_spectral')
OUT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
RUN_FINETUNE = False  # set True and rerun optional cells; baseline remains unchanged
TRAIN_PER_CLASS, VAL_PER_CLASS, EPOCHS = 120, 40, 5
S = requests.Session()
S.headers.update({'User-Agent': 'RipeSense-research/1.0'})

def save_json(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2, default=str))

def download(url, dest, max_bytes=2_000_000_000, expected_size=None):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size and (expected_size is None or dest.stat().st_size == expected_size):
        return dest
    tmp = dest.with_suffix(dest.suffix + '.part')
    try:
        with S.get(url, stream=True, timeout=(30, 180)) as r:
            r.raise_for_status()
            if int(r.headers.get('Content-Length', 0)) > max_bytes:
                raise ValueError('Download exceeds configured size limit')
            total = 0
            with tmp.open('wb') as f:
                for chunk in r.iter_content(1024 * 1024):
                    total += len(chunk)
                    if total > max_bytes: raise ValueError('Download exceeds size limit')
                    f.write(chunk)
        if expected_size is not None and total != expected_size:
            raise ValueError(f'Incomplete download: {total} != {expected_size}')
        tmp.replace(dest)
    finally:
        tmp.unlink(missing_ok=True)
    return dest

save_json('settings.json', dict(seed=SEED, run_finetune=RUN_FINETUNE,
          train_per_class=TRAIN_PER_CLASS, val_per_class=VAL_PER_CLASS, epochs=EPOCHS))
(OUT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
print('Results and models:', OUT, '\nTemporary data:', WORK)


## 1 — Pretrained detector: run today
Use the small COCO-pretrained **YOLO11n**. Supported fruit categories are apple, banana and orange; detection is not a ripeness prediction. Other fruit species may be missed or misidentified.
The public COCO fruit photograph (val2017 image 189078) is fetched directly; every detected target gets a numbered box and an unresized RGB crop. No classifier weights from Sujal are required for this step.


In [ ]:
from ultralytics import YOLO
BASELINE_WEIGHTS = WORK / 'yolo11n.pt'
baseline = YOLO(str(BASELINE_WEIGHTS))
shutil.copy2(BASELINE_WEIGHTS, OUT / 'detector_pretrained_yolo11n.pt')
FRUIT_NAMES = {'apple', 'banana', 'orange'}
FRUIT_IDS = sorted(k for k, v in baseline.names.items() if v in FRUIT_NAMES)
assert {baseline.names[k] for k in FRUIT_IDS} == FRUIT_NAMES
print('COCO model class IDs:', {k: baseline.names[k] for k in FRUIT_IDS})
EXAMPLE_URL = 'https://s3.amazonaws.com/images.cocodataset.org/val2017/000000189078.jpg'
example = download(EXAMPLE_URL, WORK / 'coco_000000189078.jpg', max_bytes=20_000_000)
save_json('demo_source.json', {'url': EXAMPLE_URL, 'coco_image_id': 189078, 'original_url': 'http://farm5.staticflickr.com/4026/4430551771_58c190b055_z.jpg', 'license': 'https://creativecommons.org/licenses/by/2.0/', 'sha256': hashlib.sha256(example.read_bytes()).hexdigest()})

def detect_and_export(image_path, model=baseline, tag='public_baseline', conf=0.25):
    """Also accepts a future phone/gantry RGB file. Box IDs are per image, not tracked identities."""
    dest = OUT / tag; dest.mkdir(parents=True, exist_ok=True)
    rgb = ImageOps.exif_transpose(Image.open(image_path)).convert('RGB')
    ids = [k for k, v in model.names.items() if v in FRUIT_NAMES]
    result = model.predict(rgb, classes=ids, conf=conf, imgsz=640, verbose=False)[0]
    canvas = rgb.copy(); draw = ImageDraw.Draw(canvas)
    records = []
    boxes = sorted(result.boxes.data.cpu().tolist(), key=lambda b: (b[0], b[1]))
    for n, (x1, y1, x2, y2, score, cls) in enumerate(boxes, 1):
        x1, y1 = max(0, int(np.floor(x1))), max(0, int(np.floor(y1)))
        x2, y2 = min(rgb.width, int(np.ceil(x2))), min(rgb.height, int(np.ceil(y2)))
        if x2 <= x1 or y2 <= y1: continue
        label = model.names[int(cls)]
        crop = f'{n:03d}_{label}.png'
        rgb.crop((x1,y1,x2,y2)).save(dest / crop)
        draw.rectangle((x1,y1,x2,y2), outline='red', width=3)
        draw.text((x1, max(0,y1-14)), f'{n}: {label} {score:.2f}', fill='white', stroke_width=2, stroke_fill='black')
        records.append(dict(detection_id=n, fruit=label, confidence=score,
                            x1=x1,y1=y1,x2=x2,y2=y2,crop=crop,source=str(image_path)))
    canvas.save(dest / 'numbered_boxes.jpg')
    pd.DataFrame(records, columns=['detection_id','fruit','confidence','x1','y1','x2','y2','crop','source']).to_csv(dest / 'detections.csv', index=False)
    display(canvas); display(pd.DataFrame(records))
    if not records: print('No target fruits detected. No crops fabricated; try another image or inspect confidence.')
    return records

public_detections = detect_and_export(example)


**Sujal handoff:** `public_baseline/detections.csv` links each numbered box to its original-resolution RGB PNG. Apply the exact resize, normalization and class order exported by Sujal's trained model; do not interpret detector confidence as ripeness confidence. The original notebook exports ONNX plus a labels JSON, but those trained files are not present here. Keep `over-ripe` separate from `spoiled`; translate spelling such as `overripe` only, never merge these meanings.

## Optional — small COCO fine-tuning subset
Downloads the official **annotations only** (~252 MB ZIP) and at most 360 train + 120 validation images, not the full COCO image archives. Uses `train2017` and `val2017` separately, retains every target bounding box in selected images, and excludes images with target crowd annotations (YOLO text boxes cannot express crowd-ignore regions). Caps are per class, with overlapping images deduplicated.

Keep the original 80-class head and original class indices to allow an identical evaluation configuration for baseline and fine-tuned weights. Only target fruit labels are used. This is a fruit-only experiment; performance on other COCO classes is not assessed. COCO pretraining already used train2017; this comparison is not evidence of unseen-domain performance. Validation is used for checkpoint selection, not a final independent test.


In [ ]:
def coco_box_to_yolo(ann, width, height, class_id):
    x,y,w,h = ann['bbox']
    x1,y1 = max(0,x),max(0,y)
    x2,y2 = min(width,x+w),min(height,y+h)
    if x2 <= x1 or y2 <= y1: raise ValueError('Degenerate COCO box')
    return [class_id,(x1+x2)/(2*width),(y1+y2)/(2*height),(x2-x1)/width,(y2-y1)/height]

def prepare_coco_subset():
    import yaml
    annzip = download('https://s3.amazonaws.com/images.cocodataset.org/annotations/annotations_trainval2017.zip',
                      WORK / 'annotations_trainval2017.zip', max_bytes=350_000_000)
    root = WORK / f'coco_fruit_{RUN_ID}'; manifests=[]; split_ids={}
    with zipfile.ZipFile(annzip) as z:
        for split, cap in [('train',TRAIN_PER_CLASS), ('val',VAL_PER_CLASS)]:
            data = json.loads(z.read(f'annotations/instances_{split}2017.json'))
            cat_to_model = {c['id']: next(k for k,v in baseline.names.items() if v == c['name'])
                            for c in data['categories'] if c['name'] in FRUIT_NAMES}
            grouped=defaultdict(list); crowd=set()
            for a in data['annotations']:
                if a['category_id'] in cat_to_model:
                    grouped[a['image_id']].append(a)
                    if a.get('iscrowd'): crowd.add(a['image_id'])
            selected=set(); rng=random.Random(SEED)
            for category in sorted(cat_to_model):
                candidates=sorted(i for i,aa in grouped.items() if i not in crowd and any(a['category_id']==category for a in aa))
                rng.shuffle(candidates); selected.update(candidates[:cap])
            split_ids[split]=selected
            image_dir=root/'images'/split; label_dir=root/'labels'/split
            image_dir.mkdir(parents=True); label_dir.mkdir(parents=True)
            for im in data['images']:
                if im['id'] not in selected: continue
                filename=im['file_name']
                download(f'https://s3.amazonaws.com/images.cocodataset.org/{split}2017/{filename}',image_dir/filename,max_bytes=25_000_000)
                with Image.open(image_dir/filename) as decoded:
                    assert decoded.size == (im['width'],im['height'])
                labels=[coco_box_to_yolo(a,im['width'],im['height'],cat_to_model[a['category_id']]) for a in grouped[im['id']]]
                (label_dir / (Path(filename).stem+'.txt')).write_text('\n'.join(' '.join(map(str,row)) for row in labels)+'\n')
                manifests.append(dict(split=split,image_id=im['id'],filename=filename,boxes=len(labels),
                                      annotation_ids=[a['id'] for a in grouped[im['id']]],
                                      sha256=hashlib.sha256((image_dir/filename).read_bytes()).hexdigest(),
                                      source=im.get('coco_url'),license_id=im.get('license')))
            save_json(f'coco_{split}_licenses.json',data['licenses'])
    assert not (split_ids['train'] & split_ids['val'])
    m=pd.DataFrame(manifests)
    assert not (set(m[m.split=='train'].sha256) & set(m[m.split=='val'].sha256)), 'Duplicate images across splits'
    m.to_csv(OUT/'coco_subset_manifest.csv',index=False)
    config=dict(path=str(root),train='images/train',val='images/val',names=baseline.names)
    yamlpath=root/'fruit.yaml'; yamlpath.write_text(yaml.safe_dump(config))
    shutil.copy2(yamlpath,OUT/'coco_fruit.yaml')
    display(m.groupby('split').agg(images=('image_id','count'),boxes=('boxes','sum')))
    return yamlpath

if RUN_FINETUNE:
    coco_yaml = prepare_coco_subset()
else:
    print('Optional download and fine-tuning skipped. Set RUN_FINETUNE=True to enable.')


In [ ]:
if RUN_FINETUNE:
    import torch
    device = 0 if torch.cuda.is_available() else 'cpu'
    eval_args=dict(data=str(coco_yaml),classes=FRUIT_IDS,imgsz=640,batch=8,device=device,plots=True)
    before=YOLO(str(OUT/'detector_pretrained_yolo11n.pt')).val(**eval_args,project=str(OUT/'evaluation'),name='baseline')
    finetuned=YOLO(str(OUT/'detector_pretrained_yolo11n.pt'))
    finetuned.train(data=str(coco_yaml),classes=FRUIT_IDS,epochs=EPOCHS,imgsz=640,batch=8,
                    device=device,seed=SEED,workers=2,project=str(OUT/'training'),name='fruit_finetune')
    best=Path(finetuned.trainer.best)
    shutil.copy2(best,OUT/'detector_finetuned_best.pt')
    finetuned=YOLO(str(OUT/'detector_finetuned_best.pt'))
    after=finetuned.val(**eval_args,project=str(OUT/'evaluation'),name='finetuned')
    comparison=pd.DataFrame([dict(model=name,**metrics.results_dict) for name,metrics in [('pretrained',before),('finetuned',after)]])
    comparison.to_csv(OUT/'detector_comparison.csv',index=False); display(comparison)
    detect_and_export(example,finetuned,'public_finetuned')


## 2 — DeepHS spectral ripeness: five fruits, no physical sensor
This independent section uses **avocado, kiwi, persimmon (`Kaki` in the source), papaya and mango** from the [official DeepHS repository](https://github.com/cogsys-tuebingen/deephs_fruit) and its [public recording/annotation server](https://cogsys.cs.uni-tuebingen.de/webprojects/DeepHS-Fruit-2023-Datasets/). You may run just this section; it does not depend on YOLO or the optional Mendeley audit below.

**Run order:** install/setup → helpers → annotation inventory/plan → process batch → train/evaluate → Try one recording.
Start with `DEEPHS_MODE='smoke'`: **nine mango recordings (~22 MB)**, one view from three independent specimens per stage; optionally set `DEEPHS_SMOKE_FRUITS` to all five fruits (~1.66 GB). Two specimens train and one is held out per stage. This demonstrates the complete pipeline; tiny smoke scores are **not reliable performance estimates**. No physical fruit or sensor is needed. Full mode transfers many GB over time; individual avocado/papaya recordings are larger than the 64×64 mango/persimmon cubes.

Then change to `DEEPHS_MODE='full'`, rerun inventory/plan, and repeatedly run the batch cell. It processes all **596 labeled Specim VIS recordings** in bounded batches. Feature rows/previews persist in Drive, so interruption loses at most the current recording. Complete the selected plan before reporting results. No 2–72 GB archive is downloaded in full. If HTTP range access fails, the cell logs the error and can be retried; it never silently downloads an entire archive.

**Matching and identity:** use the corrected `annotations-upd-2024-01-09.zip`, joining annotations to recordings inside each JSON before combining them. Use only labeled `VIS` records, not NIR, VIS_COR or inferred labels for unlabeled days. A physical specimen is `fruit + measurement series + fruit number`; day, camera and side are excluded. This follows the source's naming/measurement-series definitions and prevents recycled fruit numbers from merging different series. The coarse source labels `unripe`, `perfect`, `overripe` become `unripe`, `ripe`, `over-ripe`; fine labels remain in the manifest. There is **no spoiled label**.

The original record-level partitions are retained as provenance but replaced by a fixed, specimen-stratified 75/25 train/test split. Smoke/full plans have distinct locked splits; **do not pool their results**. This is not the published DeepHS benchmark. Keep the model configuration fixed; do not tune hyperparameters or change masks to improve held-out scores. Label assignments, counts, splits and all feature settings are saved.


In [ ]:
%pip -q install remotezip==0.12.3 numpy pandas scipy scikit-learn pillow joblib matplotlib


In [ ]:
from pathlib import Path
from google.colab import drive
from IPython.display import display
import matplotlib.pyplot as plt
import json, hashlib

drive.mount('/content/drive')
DEEPHS_MODE = 'smoke'  # 'full' uses every labeled VIS recording across all five fruits
DEEPHS_SMOKE_FRUITS = ['mango']  # optionally list all five; ignored in full mode
DEEPHS_BATCH_SIZE = 9 if DEEPHS_MODE == 'smoke' else 20
DEEPHS_ROOT = Path('/content/drive/MyDrive/RipeSense_Detection_Spectral/DeepHS')
DEEPHS_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent feature cache:', DEEPHS_ROOT)


### Feature definition and instrument approximation
The corrected annotation JSON supplies the **224 Specim FX10 wavelengths, 397.66–1003.81 nm**; ENVI headers specify array layout but the inspected headers do not contain wavelengths. The helper checks that cube channels match this documented order. Official loading code treats the published cubes as already referenced; **do not apply a second `(sample−dark)/(white−dark)` correction**.

Find a fruit region using finite pixels, mean referenced intensity across all bands between 0.05 and 1.5, the largest connected component, and one-pixel erosion. The source uses a 0.2 threshold; we lower it to 0.05 to include dark avocado skin, then use connected components and erosion. These are our adaptations, not official ground-truth masks. We omit its hard-coded border columns because some cubes are only 64 pixels wide. Preview overlays expose the mask; failed size/finite checks stop that recording. This is a heuristic, not a ground-truth segmentation. Review overlays across every fruit before trusting results, without choosing settings to improve held-out scores.

Compute the region's mean spectrum, then integrate with normalized Gaussian channel weights (including wavelength-bin widths). This gives eight **AS7341-like reflectance features**, in F1–F8 order:

| Channel | Center nm | Typical FWHM nm |
|---|---:|---:|
| F1 | 415 | 26 |
| F2 | 445 | 30 |
| F3 | 480 | 36 |
| F4 | 515 | 39 |
| F5 | 555 | 39 |
| F6 | 590 | 40 |
| F7 | 630 | 50 |
| F8 | 680 | 52 |

Centers/FWHM come from the [AS7341 datasheet, Figure 7](https://look.ams-osram.com/m/24266a3e584de4db/original/AS7341-DS000504.pdf). The Gaussian shape is **our approximation**, not measured AS7341 spectral response. F1's short-wavelength tail is truncated by the camera's lower limit and renormalized. NIR, clear and flicker are excluded. Spatial averaging, source illumination, camera referencing and AS7341 optics differ; these are not simulated ADC counts or a deployable sensor calibration.

The next cell embeds the tested helper module so this notebook works when uploaded alone; no local repo import is required. Matching source: `src/ripesense_deephs.py`.


In [ ]:
"""Standalone DeepHS feature workflow; embedded verbatim in the Colab notebook."""
from pathlib import Path
import hashlib, io, json, re, tempfile, zipfile, time
import numpy as np
import pandas as pd
import requests
from remotezip import RemoteZip
from scipy import ndimage
from PIL import Image

DEEPHS_BASE = 'https://cogsys.cs.uni-tuebingen.de/webprojects/DeepHS-Fruit-2023-Datasets/'
ANNOTATION_NAME = 'annotations-upd-2024-01-09.zip'
SOURCE_COMMIT = '5f47812a3fb5bd8f0ed5367cedf9b464f83076c1'
FRUITS = {'Avocado':'avocado', 'Kiwi':'kiwi', 'Kaki':'persimmon', 'Papaya':'papaya', 'Mango':'mango'}
STAGES = ['unripe', 'ripe', 'over-ripe']
CENTERS = np.array([415,445,480,515,555,590,630,680], dtype=float)
FWHM = np.array([26,30,36,39,39,40,50,52], dtype=float)
FEATURES = [f'F{i+1}_{int(w)}nm_mean_reflectance' for i,w in enumerate(CENTERS)]
ALGORITHM = 'deephs-vis-gaussian-fruitmask-v1'


def atomic_json(path, data):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.part')
    tmp.write_text(json.dumps(data,indent=2,allow_nan=False)); tmp.replace(path)


def get_annotations(cache):
    """Numeric record_id is local to each JSON; join before concatenating splits."""
    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)
    path=cache/ANNOTATION_NAME
    if not path.exists():
        r=requests.get(DEEPHS_BASE+ANNOTATION_NAME,timeout=90); r.raise_for_status()
        if len(r.content)>5_000_000: raise ValueError('Unexpected annotation download size')
        zipfile.ZipFile(io.BytesIO(r.content)).testzip()
        path.write_bytes(r.content)
    rows=[]; wavelengths=None
    with zipfile.ZipFile(path) as z:
        for split in ['train_only_labeled_v2','val_v2','test_v2']:
            d=json.loads(z.read('annotations/'+split+'.json'))
            wave=next(c['wavelengths'] for c in d['cameras'] if c['id']=='VIS')
            if wavelengths is not None: assert wave==wavelengths
            wavelengths=wave
            annotations={a['record_id']:a for a in d['annotations']}
            assert len(annotations)==len(d['annotations'])
            for r in d['records']:
                if r['camera_type']!='VIS' or r['fruit'] not in FRUITS or r['id'] not in annotations: continue
                a=annotations[r['id']]
                stage={'unripe':'unripe','perfect':'ripe','overripe':'over-ripe'}.get(a['ripeness_state'])
                if stage is None: raise ValueError('Unexpected coarse ripeness label')
                day=r['day']; match=re.search(r'_(\d+)_(front|back)\.bin$',r['files']['data_file'])
                if not match: raise ValueError('Unverified specimen filename: '+str(r))
                series=next((f'm{i}' for i in (2,3,4) if f'm{i}' in day),'m1')
                assert match[2]==r['side']
                specimen=f"{FRUITS[r['fruit']]}/{series}/{match[1]}"
                key=hashlib.sha256(r['files']['data_file'].encode()).hexdigest()[:24]
                rows.append(dict(key=key,fruit=FRUITS[r['fruit']],archive=r['fruit']+'.zip',
                    specimen=specimen,series=series,day=day,side=r['side'],stage=stage,
                    source_stage=a['ripeness_state'],fine_stage=a.get('ripeness_state_fine'),
                    header=r['files']['header_file'],data=r['files']['data_file'],official_split=split))
    df=pd.DataFrame(rows)
    assert df.key.is_unique and df.data.is_unique
    assert (df.groupby('specimen').stage.nunique()==1).all(), 'Conflicting specimen labels; audit first'
    wavelengths=np.asarray(wavelengths,dtype=float)
    assert len(wavelengths)==224 and np.all(np.diff(wavelengths)>0)
    source_hash=hashlib.sha256(path.read_bytes()).hexdigest()
    return df,wavelengths,source_hash


def make_plan(records, mode='smoke', seed=42, smoke_fruits=('mango',)):
    """Custom specimen-stratified split; never use a recording-level official split."""
    assert mode in ('smoke','full')
    if mode=='smoke':
        assert set(smoke_fruits) <= set(FRUITS.values()) and len(smoke_fruits)
        records=records[records.fruit.isin(smoke_fruits)]
    selected=[]; assignments={}
    for (fruit,stage),g in records.groupby(['fruit','stage'],sort=True):
        specimens=sorted(g.specimen.unique(),key=lambda s:hashlib.sha256(f'{seed}:{s}'.encode()).hexdigest())
        if mode=='smoke': specimens=specimens[:3]
        # One locked split per mode. Smoke is an intentionally tiny pipeline check.
        ntest=max(1,int(np.ceil(len(specimens)*0.25))) if len(specimens)>=2 else 0
        for i,s in enumerate(specimens): assignments[s]='test' if i<ntest else 'train'
        selected.extend(specimens)
    plan=records[records.specimen.isin(selected)].copy()
    if mode=='smoke':
        # A single view per specimen saves bandwidth; omitted views are never used in another split.
        plan=plan.sort_values(['specimen','day','side']).drop_duplicates('specimen')
    plan['split']=plan.specimen.map(assignments)
    assert (plan.groupby('specimen').split.nunique()==1).all()
    return plan.sort_values(['fruit','split','stage','specimen','day','side']).reset_index(drop=True)


def channel_weights(wavelengths):
    w=np.asarray(wavelengths,dtype=float)
    assert w.min() <= 400 and w.max() >= 710, 'Insufficient visible band coverage'
    # Gaussian approximation; wavelength-bin widths account for irregular sampling.
    weights=np.exp(-0.5*((w[:,None]-CENTERS)/(FWHM/2.35482))**2)*np.gradient(w)[:,None]
    weights /= weights.sum(axis=0,keepdims=True)
    return weights


def read_envi(header_bytes, data_bytes):
    """Read documented ENVI BIP/float32 layout, fail on an unexpected encoding."""
    meta=dict(re.findall(r'^\s*([^=\n]+?)\s*=\s*([^\n]+)',header_bytes.decode(),flags=re.M))
    meta={k.strip().lower():v.strip() for k,v in meta.items()}
    assert meta['interleave'].lower()=='bip' and int(meta['data type'])==4
    shape=tuple(int(meta[k]) for k in ('lines','samples','bands'))
    offset=int(meta.get('header offset',0)); endian='<' if int(meta['byte order'])==0 else '>'
    assert len(data_bytes)-offset==np.prod(shape)*4, 'Truncated/mismatched cube'
    return np.frombuffer(data_bytes,dtype=endian+'f4',offset=offset).reshape(shape)


def extract_features(cube,wavelengths):
    assert cube.ndim==3 and cube.shape[2]==len(wavelengths)
    # Source cubes are already referenced. No second dark/white normalization.
    valid=np.isfinite(cube).all(axis=2)
    brightness=np.mean(cube,axis=2)
    candidate=valid & (brightness>0.05) & (brightness<1.5)
    candidate=ndimage.binary_fill_holes(candidate)
    components,n=ndimage.label(candidate)
    if not n: raise ValueError('No fruit foreground above referenced mean 0.05')
    sizes=np.bincount(components.ravel()); sizes[0]=0
    mask=components==sizes.argmax()
    # Erode one pixel to reduce mixed boundary spectra; use all image sizes without fixed-column cuts.
    inner=ndimage.binary_erosion(mask,iterations=1)
    if inner.sum()>=25: mask=inner
    mask &= valid
    fraction=float(mask.mean())
    if mask.sum()<25 or not 0.02 <= fraction <= 0.98:
        raise ValueError(f'Implausible fruit region: {fraction:.3f}; inspect segmentation')
    mean_spectrum=np.mean(cube[mask],axis=0,dtype=np.float64)
    vector=mean_spectrum@channel_weights(wavelengths)
    if not np.isfinite(vector).all(): raise ValueError('Nonfinite feature')
    rgb=cube[:,:,[np.argmin(abs(wavelengths-x)) for x in (630,555,480)]]
    preview=np.clip(np.nan_to_num(rgb),0,1)**(1/2.2)
    preview=(preview*255).astype(np.uint8)
    edge=mask ^ ndimage.binary_erosion(mask)
    preview[edge]=[255,0,255]
    qc=dict(mask_fraction=fraction,mask_pixels=int(mask.sum()),shape=list(cube.shape),
            reflectance_min=float(np.min(cube[mask])),reflectance_max=float(np.max(cube[mask])))
    return vector,preview,qc


def feature_signature(wavelengths,annotation_hash):
    spec=dict(algorithm=ALGORITHM,annotation_sha256=annotation_hash,source_commit=SOURCE_COMMIT,
              wavelengths_nm=list(map(float,wavelengths)),centers_nm=CENTERS.tolist(),fwhm_nm=FWHM.tolist(),
              feature_order=FEATURES,mask='finite; 0.05 < mean(all224bands) < 1.5; largest component; erode1',
              calibration='already-referenced DeepHS cubes; no second correction',sensor_ready=False)
    digest=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()[:16]
    return digest,spec


def process_batch(plan,wavelengths,cache,limit=15,max_member_mb=500):
    """Commit after each recording. No full archive downloads; missing rows retry next time."""
    cache=Path(cache); (cache/'rows').mkdir(parents=True,exist_ok=True); (cache/'previews').mkdir(exist_ok=True)
    pending=plan[~plan.key.map(lambda k:(cache/'rows'/f'{k}.json').exists())].head(limit)
    completed=0; failures=[]
    for archive,g in pending.groupby('archive',sort=False):
        try:
            with RemoteZip(DEEPHS_BASE+archive,timeout=180) as z:
                for r in g.to_dict('records'):
                    try:
                        hi,di=z.getinfo(r['header']),z.getinfo(r['data'])
                        if max(hi.file_size,di.file_size)>max_member_mb*1_000_000:
                            raise ValueError('Member exceeds RAM guard; increase only after checking Colab memory')
                        # RemoteZip validates the member CRC during read.
                        h=z.read(r['header']); b=z.read(r['data'])
                        cube=read_envi(h,b)
                        vector,preview,qc=extract_features(cube,wavelengths)
                        Image.fromarray(preview).save(cache/'previews'/f"{r['key']}.png")
                        row={**r,**dict(zip(FEATURES,map(float,vector))),**qc,
                             'data_sha256':hashlib.sha256(b).hexdigest(),
                             'zip_crc':di.CRC,'compressed_bytes':di.compress_size}
                        atomic_json(cache/'rows'/f"{r['key']}.json",row)
                        del cube,b,preview
                        completed+=1; print(f"Saved {r['fruit']} {r['specimen']} {r['side']} ({completed}/{len(pending)})",flush=True)
                    except Exception as e:
                        failures.append(dict(key=r['key'],data=r['data'],error=repr(e)))
                        print('FAILED',r['data'],repr(e),flush=True)
        except Exception as e:
            failures.append(dict(archive=archive,error=repr(e)))
            print('ARCHIVE FAILED',archive,repr(e),flush=True)
    atomic_json(cache/'last_batch.json',dict(attempted=len(pending),completed=completed,failures=failures))
    return completed


def load_features(plan,cache):
    rows=[]
    for r in plan.to_dict('records'):
        p=Path(cache)/'rows'/f"{r['key']}.json"
        if p.exists():
            row=json.loads(p.read_text())
            # Split belongs to this plan, not a cached row from another mode.
            assert row['specimen']==r['specimen'] and row['stage']==r['stage']
            row['split']=r['split']; rows.append(row)
    return pd.DataFrame(rows)


def train_and_report(plan,features,output,mode):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score,balanced_accuracy_score,confusion_matrix
    import joblib
    output=Path(output); output.mkdir(parents=True,exist_ok=True)
    if features.empty or set(plan.key)!=set(features.key):
        raise ValueError('Finish every planned recording before fitting/reporting. Rerun batches; failures are not silently dropped.')
    assert (features.groupby('specimen').split.nunique()==1).all()
    assert set(features[features.split=='train'].specimen).isdisjoint(features[features.split=='test'].specimen)
    assert np.isfinite(features[FEATURES].to_numpy()).all()
    models={}; summaries=[]; stage_rows=[]; predictions=[]
    for fruit in FRUITS.values():
        f=features[features.fruit==fruit]; tr=f[f.split=='train']; te=f[f.split=='test']
        train_counts=tr.groupby('stage').specimen.nunique(); test_counts=te.groupby('stage').specimen.nunique()
        if tr.stage.nunique()<2 or te.empty:
            summaries.append(dict(fruit=fruit,status='not selected in this plan' if f.empty else 'insufficient training classes or held-out specimens'))
            for stage in STAGES:
                stage_rows.append(dict(fruit=fruit,stage=stage,train_specimens=int(train_counts.get(stage,0)),
                    test_specimens=int(test_counts.get(stage,0)),status='not evaluable in this plan',recall=None))
            continue
        model=make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=2000,random_state=42))
        # Each physical specimen gets total training weight 1, irrespective of views/days.
        weights=1/tr.specimen.map(tr.groupby('specimen').size()).to_numpy()
        model.fit(tr[FEATURES],tr.stage,logisticregression__sample_weight=weights)
        models[fruit]=model
        prob=model.predict_proba(te[FEATURES]); classes=model.classes_
        rec=te[['key','fruit','specimen','day','side','stage']].copy()
        rec['predicted']=classes[prob.argmax(axis=1)]
        rec['max_probability']=prob.max(axis=1)
        for j,label in enumerate(classes): rec['p_'+label]=prob[:,j]
        predictions.append(rec)
        # Aggregate all held-out views to one prediction per physical fruit.
        specimen_prob=rec.groupby('specimen')[['p_'+s for s in classes]].mean()
        true=rec.groupby('specimen').stage.first().loc[specimen_prob.index]
        pred=classes[specimen_prob.to_numpy().argmax(axis=1)]
        specimen_out=pd.DataFrame(dict(specimen=specimen_prob.index,stage=true.to_numpy(),predicted=pred))
        specimen_out.to_csv(output/f'{fruit}_heldout_specimens.csv',index=False)
        for stage in STAGES:
            nt,nv=int(train_counts.get(stage,0)),int(test_counts.get(stage,0))
            enough=nt>=2 and nv>=2
            status='evaluated (small sample)' if enough else 'too few independent specimens (need >=2 train and >=2 test)'
            selected=true.to_numpy()==stage
            stage_rows.append(dict(fruit=fruit,stage=stage,train_specimens=nt,test_specimens=nv,status=status,
                recall=float(np.mean(pred[selected]==stage)) if enough else None))
        summaries.append(dict(fruit=fruit,status='SMOKE ONLY; not a performance estimate' if mode=='smoke' else 'held-out specimen evaluation',
            train_specimens=int(tr.specimen.nunique()),test_specimens=int(te.specimen.nunique()),
            specimen_accuracy=float(accuracy_score(true,pred)),specimen_balanced_accuracy=float(balanced_accuracy_score(true,pred)),
            recording_accuracy=float(accuracy_score(rec.stage,rec.predicted)),
            confusion_matrix=confusion_matrix(true,pred,labels=STAGES).tolist(),confusion_order=STAGES))
    bundle=dict(models=models,feature_order=FEATURES,algorithm=ALGORITHM,mode=mode,stages=STAGES,sensor_ready=False)
    joblib.dump(bundle,output/'deephs_ripeness.joblib')
    atomic_json(output/'feature_order.json',dict(feature_order=FEATURES,centers_nm=CENTERS.tolist(),fwhm_nm=FWHM.tolist(),sensor_ready=False))
    atomic_json(output/'heldout_results.json',summaries)
    pd.DataFrame(stage_rows).to_csv(output/'stage_support.csv',index=False)
    predframe=pd.concat(predictions,ignore_index=True) if predictions else pd.DataFrame()
    predframe.to_csv(output/'heldout_recordings.csv',index=False)
    features.to_csv(output/'features_and_split.csv',index=False)
    return bundle,pd.DataFrame(summaries),pd.DataFrame(stage_rows),predframe


In [ ]:
deephs_records, deephs_wavelengths, deephs_annotation_sha = get_annotations(DEEPHS_ROOT / 'metadata')
signature, feature_schema = feature_signature(deephs_wavelengths, deephs_annotation_sha)
DEEPHS_CACHE = DEEPHS_ROOT / ('features_' + signature)
DEEPHS_RESULTS = DEEPHS_ROOT / ('results_' + signature) / (DEEPHS_MODE + ('_' + '-'.join(sorted(DEEPHS_SMOKE_FRUITS)) if DEEPHS_MODE == 'smoke' else ''))
DEEPHS_RESULTS.mkdir(parents=True, exist_ok=True)
atomic_json(DEEPHS_RESULTS / 'feature_schema.json', feature_schema)
deephs_records.to_csv(DEEPHS_RESULTS / 'all_labeled_vis_manifest.csv', index=False)
print('Documented VIS wavelengths:', len(deephs_wavelengths), 'bands; nm range:', deephs_wavelengths[[0,-1]])
print('First wavelengths:', deephs_wavelengths[:12])
display(deephs_records[['fruit','specimen','day','side','stage','fine_stage','data']].head(10))
counts = deephs_records.groupby(['fruit','stage']).specimen.nunique().unstack(fill_value=0).reindex(columns=STAGES,fill_value=0)
print('Independent physical specimens, all labeled VIS data:')
display(counts); counts.to_csv(DEEPHS_RESULTS / 'independent_specimen_counts.csv')
deephs_plan = make_plan(deephs_records, DEEPHS_MODE, smoke_fruits=DEEPHS_SMOKE_FRUITS)
plan_file = DEEPHS_RESULTS / 'locked_plan.json'
plan_rows = deephs_plan.to_dict('records')
if plan_file.exists():
    assert json.loads(plan_file.read_text()) == plan_rows, 'Plan changed: use a new results directory; do not overwrite the held-out split.'
else:
    atomic_json(plan_file, plan_rows)
deephs_plan.to_csv(DEEPHS_RESULTS / 'split_manifest.csv', index=False)
display(deephs_plan.groupby(['fruit','stage','split']).specimen.nunique().rename('specimens').unstack(fill_value=0))
print(DEEPHS_MODE, ':', len(deephs_plan), 'planned recordings;', deephs_plan.specimen.nunique(), 'independent specimens')
print('Feature cache:', DEEPHS_CACHE, '\nReports/models:', DEEPHS_RESULTS)


In [ ]:
# Re-run this cell until remaining == 0. Existing verified feature rows are skipped.
processed = process_batch(deephs_plan, deephs_wavelengths, DEEPHS_CACHE, limit=DEEPHS_BATCH_SIZE)
deephs_features = load_features(deephs_plan, DEEPHS_CACHE)
remaining = len(deephs_plan) - len(deephs_features)
print('Saved in this batch:', processed, '| total cached:', len(deephs_features), '| remaining:', remaining)
if remaining:
    print('Run this cell again. Inspect last_batch.json if any recordings failed; training requires the entire plan.')
# Persistent PNGs make mask quality review possible without re-downloading cubes.
if not deephs_features.empty:
    examples = deephs_features.groupby('fruit',sort=False).head(1)
    fig,axes=plt.subplots(1,len(examples),figsize=(4*len(examples),4),squeeze=False)
    for ax,r in zip(axes.flat,examples.to_dict('records')):
        ax.imshow(Image.open(DEEPHS_CACHE/'previews'/f"{r['key']}.png"))
        ax.set_title(f"{r['fruit']} — mask {r['mask_fraction']:.1%}"); ax.axis('off')
    plt.tight_layout(); plt.show()


### Fit and held-out evaluation
Fit one small standardized logistic-regression model per fruit with a fixed configuration. Fruit type is selected explicitly, not inferred by the spectral model. Every physical specimen receives total training weight 1 regardless of the number of views. All scaling is learned on training rows only.

Report held-out predictions per recording **and** per specimen (average view probabilities first). The specimen score is primary. Print confusion matrices and support per stage. A stage needs at least two training and two held-out specimens for the displayed stage recall; otherwise it is marked **too few to evaluate**. Even two test specimens give very uncertain estimates. Smoke scores are pipeline diagnostics only. Repeated days/views are never counted as independent test fruit.


In [ ]:
deephs_features = load_features(deephs_plan, DEEPHS_CACHE)
deephs_bundle, deephs_summary, deephs_stage_support, deephs_predictions = train_and_report(
    deephs_plan, deephs_features, DEEPHS_RESULTS, DEEPHS_MODE)
display(deephs_summary.drop(columns=['confusion_matrix','confusion_order'],errors='ignore'))
display(deephs_stage_support)
for result in deephs_summary.to_dict('records'):
    if isinstance(result.get('confusion_matrix'), list):
        print(result['fruit'], '— held-out physical specimens; rows=true, columns=predicted')
        display(pd.DataFrame(result['confusion_matrix'],index=STAGES,columns=STAGES))
print('Saved model bundle and exact feature order:', DEEPHS_RESULTS)


### Try one recording
Select a held-out fruit and recording index below. The cell loads the saved model and feature order from Drive, uses a cached feature row, shows the pseudo-RGB preview with the fruit-mask boundary in magenta, and compares prediction with the matched dataset label. It refuses training specimens. The pseudo-RGB image is a display rendering, not the model input. Probabilities are not calibrated confidence or food-safety estimates.


In [ ]:
TRY_FRUIT = 'mango'  # @param ['avocado', 'kiwi', 'persimmon', 'papaya', 'mango']
TRY_INDEX = 0  # @param {type:'integer'}
import joblib
saved_bundle = joblib.load(DEEPHS_RESULTS / 'deephs_ripeness.joblib')
assert saved_bundle['feature_order'] == FEATURES
heldout = deephs_features[(deephs_features.fruit==TRY_FRUIT)&(deephs_features.split=='test')].sort_values('key').reset_index(drop=True)
assert len(heldout), 'No held-out recording for this fruit; finish the plan first.'
assert 0 <= TRY_INDEX < len(heldout), f'Choose an index from 0 to {len(heldout)-1}'
chosen = heldout.iloc[[TRY_INDEX]]
assert chosen.specimen.iloc[0] not in set(deephs_features[deephs_features.split=='train'].specimen)
model = saved_bundle['models'][TRY_FRUIT]
probabilities = model.predict_proba(chosen[saved_bundle['feature_order']])[0]
prediction = model.classes_[probabilities.argmax()]
preview_path = DEEPHS_CACHE/'previews'/f"{chosen.key.iloc[0]}.png"
display(Image.open(preview_path))
print('Specimen:', chosen.specimen.iloc[0], '| day:', chosen.day.iloc[0], '| side:', chosen.side.iloc[0])
print('Dataset label:', chosen.stage.iloc[0], '| predicted:', prediction,
      '| match:', bool(prediction==chosen.stage.iloc[0]))
display(pd.DataFrame({'stage':model.classes_,'probability':probabilities}))
print('Recording:', chosen.data.iloc[0])


### Before connecting an AS7341
This model validates a public-recording feature workflow, not sensor transfer. Measure dark and white references under the actual illumination; control LED spectrum, distance, field of view, integration time, gain, diffuser and saturation. Verify F1–F8 order and units, use measured channel-response curves in place of Gaussian approximations where available, and account for the truncated F1 tail. The camera uses a whole-fruit region while a gantry sensor may see a small spot; assess multiple locations/views.

Collect paired AS7341 and reference measurements on **new labeled specimens of each fruit**, fit a transfer calibration or retrain with sensor-specific features, and evaluate on completely held-out physical specimens and acquisition sessions. Do not input raw counts to this model or use the three reserved demo apples to claim validation. Ripeness, over-ripeness and spoilage are distinct; no spoiled classifier or food-safety claim is provided.

**Provenance:** [DeepHS source pinned at 5f47812](https://github.com/cogsys-tuebingen/deephs_fruit/tree/5f47812a3fb5bd8f0ed5367cedf9b464f83076c1), especially `core/name_convention.py`, `core/fruit_list.py`, `core/spectral_io.py`, `core/util.py` and `core/datasets/hyperspectral_dataset.py`; [corrected annotations notice](https://cogsys.cs.uni-tuebingen.de/webprojects/DeepHS-Fruit-2023-Datasets/annotations-upd-2024-01-09.ReadMe). Cite Varga, Makowski & Zell, *Measuring the Ripeness of Fruit with Hyperspectral Imaging and Deep Learning*, IJCNN 2021, DOI 10.1109/IJCNN52387.2021.9533728. Review upstream data/software terms before redistribution. Downloaded recordings remain outside Git; only derived features, previews, provenance and models go to Drive.

The Mendeley section below remains a separate optional audit. Its missing metadata does not block this DeepHS workflow. Actual local tests and Colab limitations are recorded in `docs/deephs_validation.md`.


## Optional — Inspect Mendeley ripeness files before defining spectral features
[Gaikwad & Kothari, Mendeley Data v2](https://doi.org/10.17632/y5h6v8w6ms.2), CC BY 4.0. Only `Ripeness Grading.zip` is accessed. We inspect its central directory and a nine-image observation using HTTP range reads. If the server stops supporting ranges, the explicit fallback downloads the 1.56 GB ripeness ZIP to temporary Colab storage. No other dataset portions are downloaded.

**Verified during development (2026-11-07):** 3,231 JPEGs, 18 day folders, five variety folders and four apple-number folders per variety. Day 9 has 171 images; other days have 180. Example: `Ripeness Grading/day 1/kinnaur/APPLE 1/apple1_1.jpg`. The nine downloaded sample JPEGs are **17,280 × 1,080 concatenated strips**, each visually containing nine 1,920-pixel-wide panels (one dark-looking panel and eight illuminated panels). This is layout evidence, not a verified wavelength/reference key. There are no metadata tables or scripts inside this ZIP. The publisher lists under-ripe, ripe, over-ripe and eight illumination bands, but the archive has filename suffixes 1–9, no wavelength values and no stage folders. The numbers must not be treated as nanometres or assigned to AS7341 channels.


In [ ]:
from remotezip import RemoteZip
MD_API='https://data.mendeley.com/public-api/datasets/y5h6v8w6ms'
r=S.get(MD_API+'/files',params={'folder_id':'root','version':2,'$start':0,'$limit':1000},
        headers={'Accept':'application/vnd.mendeley-public-dataset.1+json'},timeout=60)
r.raise_for_status(); md_files=r.json()
ripeness=[f for f in md_files if f['filename'].lower()=='ripeness grading.zip']
assert len(ripeness)==1, 'Ripeness archive changed: inspect metadata before continuing'
meta=ripeness[0]; details=meta['content_details']; archive_url=details['download_url']
save_json('mendeley_file_metadata.json',meta)
LOCAL_ARCHIVE=None
try:
    with RemoteZip(archive_url,timeout=120) as z:
        members=z.namelist()
except Exception as e:
    print('Range inventory unavailable:',repr(e),'— downloading ripeness ZIP only.')
    LOCAL_ARCHIVE=download(archive_url,WORK/'Ripeness Grading.zip',expected_size=details['size'])
    assert hashlib.file_digest(LOCAL_ARCHIVE.open('rb'),'sha256').hexdigest()==details['sha256_hash']
    with zipfile.ZipFile(LOCAL_ARCHIVE) as z: members=z.namelist()
save_json('mendeley_archive_members.json',members)
files_in_zip=[n for n in members if not n.endswith('/')]
print('File types:',Counter(Path(n).suffix.lower() for n in files_in_zip))
print('Example filenames:'); print('\n'.join(files_in_zip[:12]))
print('Metadata files:',[n for n in files_in_zip if Path(n).suffix.lower()!='.jpg'])


In [ ]:
PATTERN=re.compile(r'^Ripeness Grading/day (\d+)/([^/]+)/APPLE (\d+)/apple(\d+)[_-](\d+)\.jpg$',re.I)
rows=[]; unmatched=[]
for name in files_in_zip:
    m=PATTERN.fullmatch(name)
    if not m: unmatched.append(name); continue
    day,variety,apple,file_apple,capture=m.groups()
    assert apple==file_apple, f'Conflicting apple numbers: {name}'
    rows.append(dict(member=name,day=int(day),variety=variety.lower(),apple_number=int(apple),
                     candidate_specimen=f'{variety.lower()}/apple_{apple}',capture_index=int(capture),
                     wavelength_nm=None,stage=None))
inventory=pd.DataFrame(rows)
inventory.to_csv(OUT/'spectral_inventory_UNVERIFIED.csv',index=False)
display(inventory.head(12))
display(inventory.groupby('day').size().rename('image_count').to_frame())
print('Candidate variety + apple IDs:',inventory.candidate_specimen.nunique())
print('Capture indices (NOT verified wavelength bands):',sorted(inventory.capture_index.unique()))
print('Unparsed members:',unmatched[:20])
counts=pd.DataFrame({'stage':['under-ripe','ripe','over-ripe','spoiled'],
                     'verified_identifiable_physical_apples':['unknown','unknown','unknown','not supplied'],
                     'reason':['No per-observation stage key']*3+['Not a dataset class']})
display(counts); counts.to_csv(OUT/'physical_apples_per_class.csv',index=False)
issues=[
    'Need author-confirmed concatenated panel -> wavelength (nm), bandwidth, black-reference and filename suffix/view mapping.',
    'Need stage label for each specimen/day/view; days are not stage ground truth.',
    'Need confirmation that variety + apple number denotes the same physical apple across all days and views.',
    'Need acquisition/exposure and preprocessing details; JPEG intensity is not calibrated reflectance.'
]
save_json('spectral_status.json',dict(status='BLOCKED_UNVERIFIED_MAPPING',accuracy=None,
          candidate_specimens=int(inventory.candidate_specimen.nunique()),issues=issues))
print('NO SPECTRAL ACCURACY REPORTED.'); print('\n'.join(issues))


In [ ]:
def read_archive_members(names):
    """Read selected members only; cache bytes under a hash, never extract untrusted paths."""
    cached={}; missing=[]
    cache=WORK/'spectral_samples'; cache.mkdir(exist_ok=True)
    for name in names:
        p=cache/(hashlib.sha256(name.encode()).hexdigest()+'.jpg')
        if p.exists(): cached[name]=p.read_bytes()
        else: missing.append((name,p))
    if missing:
        archive=zipfile.ZipFile(LOCAL_ARCHIVE) if LOCAL_ARCHIVE else RemoteZip(archive_url,timeout=120)
        with archive as z:
            for name,p in missing:
                b=z.read(name); p.write_bytes(b); cached[name]=b
    return cached

sample_names=inventory[(inventory.day==1)&(inventory.variety=='kinnaur')&(inventory.apple_number==1)].sort_values('capture_index').member.tolist()
samples=read_archive_members(sample_names)
sample_info=[]
for name in sample_names:
    im=Image.open(io.BytesIO(samples[name]))
    sample_info.append(dict(member=name,width=im.width,height=im.height,mode=im.mode))
display(pd.DataFrame(sample_info))
pd.DataFrame(sample_info).to_csv(OUT/'spectral_sample_dimensions.csv',index=False)
im=Image.open(io.BytesIO(samples[sample_names[0]]))
print('Full strip:',sample_names[0],im.size)
display(im.resize((1440,90)))
# Diagnostic equal-width slicing only. Never use these positions as a band key.
if im.size == (17280,1080):
    fig,axes=plt.subplots(3,3,figsize=(12,8))
    for i,ax in enumerate(axes.flat):
        ax.imshow(im.crop((1920*i,0,1920*(i+1),1080)))
        ax.set_title(f'Panel {i+1}: wavelength unknown'); ax.axis('off')
    plt.tight_layout(); fig.savefig(OUT/'spectral_panel_contact_sheet.png'); plt.show()
else:
    print('Layout changed; inspect manually before slicing.')
print('Concatenation suggests within-file pairing, but wavelengths, stages and persistent specimen identity remain unverified.')


### Conditional training path — only after the missing mappings are obtained
The current public archive cannot support a defensible spectral classifier by itself. This default run finishes with an audit, **not a fabricated accuracy or model**. Do not sort unrelated files and zip them into spectral rows, label early/late days by assumption, or split individual images randomly.

The cell below is a future, explicitly gated training implementation. An author-verified manifest must contain one row per verified wavelength panel with `member, specimen_id, observation_id, stage, wavelength_nm, evidence_url, x1, y1, x2, y2` (panel bounds in its source JPEG). An observation is one specimen at one time and view. Specimen IDs must persist across days, stages and views. Exclude black/reference images from the feature rows after documenting their identity. Supply the verified band order and provenance JSON as well. Neither file is required for today's detection/audit run.

Feature: median grayscale JPEG intensity inside a manually reviewed fruit mask for each band (no whole-image mean). These are **camera/illumination features**, not calibrated AS7341 measurements. Masks must be prepared independently of labels; save them as 0/255 PNGs with the cropped panel dimensions. The mask manifest supplies `mask_path` for each row. Manual review is required because the fruit sits among background/chamber objects and an automatic threshold is not established here.


In [ ]:
# Leave unset today. Later place author-verified manifest/provenance and reviewed masks in Drive.
VERIFIED_MANIFEST = None
VERIFIED_PROVENANCE = None

def train_verified_spectral(manifest_path, provenance_path):
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, accuracy_score
    import joblib
    provenance=json.loads(Path(provenance_path).read_text())
    required_evidence=['identity_across_days_and_views','stage_mapping','panel_to_wavelength_mapping','mask_review']
    assert all(provenance.get(k) for k in required_evidence), 'Missing evidence references'
    bands=[float(x) for x in provenance['band_order_nm']]
    assert len(bands)==8 and len(set(bands))==8, 'Verify all eight bands and exclude reference captures'
    m=pd.read_csv(manifest_path)
    required=['member','specimen_id','observation_id','stage','wavelength_nm','evidence_url','mask_path','x1','y1','x2','y2']
    assert set(required)<=set(m.columns)
    assert not m[required].isna().any().any()
    assert set(m.member)<=set(files_in_zip)
    assert not m.duplicated(['member','x1','y1','x2','y2']).any(), 'Repeated panel in manifest'
    assert set(m.stage)<= {'under-ripe','ripe','over-ripe'}, 'Do not relabel over-ripe as spoiled'
    assert not m.duplicated(['observation_id','wavelength_nm']).any()
    for _,g in m.groupby('observation_id'):
        assert g.specimen_id.nunique()==g.stage.nunique()==1
        assert set(g.wavelength_nm.astype(float))==set(bands), 'Incomplete/mismatched observation'
    # Keep all repeated days and views of a specimen together, even if its stage changes.
    blobs=read_archive_members(m.member.tolist()); feature_rows=[]
    for obs,g in m.groupby('observation_id',sort=True):
        row=dict(observation_id=obs,specimen_id=g.specimen_id.iloc[0],stage=g.stage.iloc[0])
        for r in g.itertuples():
            im=Image.open(io.BytesIO(blobs[r.member])).convert('L')
            bounds=[r.x1,r.y1,r.x2,r.y2]
            assert all(float(v).is_integer() for v in bounds)
            x1,y1,x2,y2=map(int,bounds)
            assert 0 <= x1 < x2 <= im.width and 0 <= y1 < y2 <= im.height
            a=np.asarray(im.crop((x1,y1,x2,y2)),dtype=float)
            mask=np.asarray(Image.open(r.mask_path).convert('L'))>127
            assert mask.shape==a.shape and 0.01 < mask.mean() < 0.95, 'Invalid fruit mask'
            row[f'nm_{float(r.wavelength_nm):g}']=float(np.median(a[mask])/255.)
        feature_rows.append(row)
    features=pd.DataFrame(feature_rows); columns=[f'nm_{b:g}' for b in bands]
    assert np.isfinite(features[columns].to_numpy()).all()
    # One prespecified split. Never search random seeds for a better score.
    train_i,val_i=next(GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=SEED).split(features,groups=features.specimen_id))
    tr,va=features.iloc[train_i],features.iloc[val_i]
    assert set(tr.specimen_id).isdisjoint(va.specimen_id)
    assert set(tr.stage)==set(va.stage)==set(features.stage), 'Too few independent specimens for this split; collect more data'
    features['split']='train'; features.loc[val_i,'split']='validation'
    features.to_csv(OUT/'spectral_features_and_split.csv',index=False)
    display(features.groupby(['split','stage']).specimen_id.nunique().rename('physical_apples').to_frame())
    model=make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=2000,random_state=SEED))
    model.fit(tr[columns],tr.stage); pred=model.predict(va[columns])
    labels=list(model.classes_)
    metrics=dict(accuracy=accuracy_score(va.stage,pred),balanced_accuracy=balanced_accuracy_score(va.stage,pred),
                 report=classification_report(va.stage,pred,output_dict=True,zero_division=0),
                 confusion_matrix=confusion_matrix(va.stage,pred,labels=labels).tolist(),labels=labels,
                 validation_specimens=va.specimen_id.nunique(),validation_observations=len(va),
                 limitation='Observation-level metrics on a grouped small holdout; correlated days/views are not independent test specimens.')
    save_json('spectral_metrics.json',metrics)
    save_json('spectral_provenance.json',provenance)
    save_json('spectral_schema.json',dict(features=columns,labels=labels,feature='fruit-mask median grayscale JPEG / 255',
                                        sensor='camera under controlled illumination; NOT AS7341',spoiled_supported=False))
    shutil.copy2(manifest_path,OUT/'verified_spectral_manifest.csv')
    joblib.dump(model,OUT/'spectral_classifier.joblib')
    save_json('spectral_status.json',dict(status='TRAINED_WITH_SUPPLIED_VERIFIED_METADATA',metrics_file='spectral_metrics.json',as7341_ready=False,spoiled_supported=False))
    display(pd.DataFrame(metrics['report']).T)
    return model

if VERIFIED_MANIFEST is not None and VERIFIED_PROVENANCE is not None:
    spectral_model=train_verified_spectral(VERIFIED_MANIFEST,VERIFIED_PROVENANCE)
else:
    print('Spectral training correctly skipped: unverified specimen, stage and wavelength mappings. Accuracy: not available.')


## Demo handoff and run limits

| Output | Runs now | Later input |
|---|---|---|
| YOLO baseline, numbered boxes, PNG crops + CSV | Public fruit image, CPU or GPU | Phone/gantry RGB photo via `detect_and_export(path, tag='live_photo')`; EXIF orientation corrected |
| Optional fine-tuned detector + comparison | Public COCO subset, preferably T4 | Same RGB path; compare before adopting |
| Spectral inventory + contact sheet + missing-mapping report | Public Mendeley archive, no hardware | Author-confirmed capture/stage/specimen metadata |
| Mendeley spectral classifier | Blocked with current public metadata | Verified paired band images + reviewed masks; grouped holdout |
| Sujal's RGB ripeness predictions | Requires his separately trained ONNX + label file | Feed exported crops using his exact preprocessing |

Optional phone upload (run manually):
```python
from google.colab import files
uploaded = files.upload()
for i, name in enumerate(uploaded):
    detect_and_export(name, tag=f'phone_{i}')
```

**AS7341 gate:** Its eight visible channels, NIR and clear channels measure a different instrument response from webcam JPEGs under switched illumination. Confirm actual channel response curves/bandwidths and ordering from the [manufacturer documentation](https://ams-osram.com/products/sensors/ambient-light-color-spectral-proximity-sensors/ams-as7341-11-channel-spectral-color-sensor). Match illumination geometry, distance, gain/integration time, dark/white references, saturation handling and normalization. Matching nominal wavelengths alone is insufficient. Obtain paired camera/AS7341 measurements on additional, independently labeled apples, fit/validate a transfer calibration or retrain a sensor-specific model, and test on held-out physical apples. Do not feed raw AS7341 counts into this camera model. An ordinary phone photo is also not an eight-band observation.

Reserve all three physical apples for the live demonstration. The rotten apple is a **spoiled** example, not an `over-ripe` training label. Mendeley supplies no verified spoiled class, so a spectral model trained on it cannot certify spoilage or food safety; display unsupported/unknown for that task.

**Storage:** Large data and caches stay under `/content`; Drive stores models, small results, provenance and crop outputs. Download/cache failures are surfaced, not silently turned into empty training data. Temporary Colab data disappears on runtime reset. Each new run has its own Drive directory.

Sources: [Ultralytics detection API](https://docs.ultralytics.com/tasks/detect/), [COCO](https://cocodataset.org/#download) (retain image-specific licenses in exported metadata), [COCO demo image source on Flickr](https://www.flickr.com/photo.gne?id=4430551771) (COCO license ID 4: CC BY 2.0; see original image attribution), [Mendeley v2, Gaikwad & Kothari](https://doi.org/10.17632/y5h6v8w6ms.2) (CC BY 4.0; derived inventories/contact sheets are our modifications). Check source/software licensing for redistribution.

**Testing:** See repository `docs/detection_spectral_validation.md` for actual development checks. Colab Drive authorization and GPU fine-tuning must be run in your own Colab session. No spectral accuracy is claimed.


In [ ]:
print('Run complete. Drive artifacts:')
for p in sorted(OUT.rglob('*')):
    if p.is_file(): print(p.relative_to(OUT))
